# 15. Pestaña «Modelos base»: interpretación de cada resultado

La pestaña compara tres modelos evaluados **una sola vez** en la prueba cronológica
(2017-04-22 a 2018-01-31). Se organiza en cinco **subpestañas**: Desempeño, ROC, Matriz de confusión, Umbral y Calibración y diagnóstico.

> **Qué muestra el dashboard y qué muestra este capítulo.** El dashboard presenta **solo el ROC-AUC** (métrica principal), con su IC 95 %, su valor en validación cruzada temporal y la comparación con el Dummy; el umbral y la matriz de confusión se muestran sin métricas adicionales.
> Este capítulo conserva la evaluación completa (Accuracy, Precision, Recall, F1, Balanced Accuracy, Average Precision y Brier, con sus IC) porque es la explicación detallada; esos valores no cambian.

1. **Dummy — clase mayoritaria / prior:** línea base trivial; predice siempre la clase mayoritaria («tardía») y da a todos la misma probabilidad (la prevalencia).
2. **Logística aditiva (modo + hora):** modo de envío y seno/coseno de la hora, sin interacción.
3. **Logística modo × jornada:** el modelo base elegido en el capítulo 10 (8 categorías).

Se eligió **regresión logística** y no SVR lineal porque el objetivo es binario (ruta A): la SVR
lineal es un modelo de **regresión** y no entrega probabilidades calibradas, que aquí son necesarias
para la curva ROC, la calibración y el umbral. No se agregan modelos nuevos: este entregable es de **modelos base**.

Cada resultado indica su origen: **TRAIN** (entrenamiento), **OOF** (predicciones fuera de pliegue del entrenamiento) o **TEST** (prueba cronológica).

In [1]:
from pathlib import Path
import re, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display

project_root = Path.cwd().resolve()
while not (project_root / "_config.yml").exists():
    if project_root.parent == project_root:
        raise FileNotFoundError("No se encontró _config.yml. Ejecuta el notebook dentro de la carpeta del libro.")
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

for _folder in ("tables", "figures"):          # se crean si no existen
    (project_root / _folder).mkdir(exist_ok=True)

from src.project_utils import *

configure_plots()
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 170)
pd.set_option("display.max_colwidth", 70)
TABLES = project_root / "tables"

def md_out(text):
    display(Markdown(text))

import plotly.io as pio
from IPython.display import HTML
sys.path.insert(0, str(project_root / "dashboard"))
import figuras as F
import textos as T
_plotly = {"cargado": False}

def mostrar(fig):
    """Muestra una figura Plotly del dashboard (la primera de la página carga plotly.js desde la CDN)."""
    incluir = "cdn" if not _plotly["cargado"] else False
    _plotly["cargado"] = True
    display(HTML(pio.to_html(fig, full_html=False, include_plotlyjs=incluir, config={"displaylogo": False, "responsive": True})))

from sklearn import metrics as skm
pred = F.predicciones()
TAB = project_root / "tables"
from src.project_utils import paired_cluster_bootstrap, classification_metrics
oof = F.predicciones_oof()
A, B_ = F.MODELO_ELEGIDO, "Logística aditiva (modo + hora)"


## 15.1 Resumen: tabla comparativa (TEST, IC 95 % por bootstrap agrupado por pedido)

In [2]:
display(HTML(F.tabla_modelos().to_html(escape=False, index=False).replace("\n", "<br>")))
md_out("*" + T.IC_NOTA + " Debajo de cada valor, el IC 95 % (prueba) o la desviación entre pliegues (CV temporal).*")
pr = F.tabla("modelos_prueba.csv")
display(pr[["modelo", "C"]].rename(columns={"C": "C elegido por CV temporal (1/regularización)"}))
md_out(T.DUMMY_NOTA)

Modelo,ROC-AUC,Accuracy,Precision,Recall,F1,Balanced Accuracy,Average Precision,Brier,ROC-AUC CV temporal
Dummy — clase mayoritaria / prior,"0,500\n[0,500–0,500]","0,573\n[0,564–0,582]","0,573\n[0,564–0,582]","1,000\n[1,000–1,000]","0,729\n[0,721–0,736]","0,500\n[0,500–0,500]","0,573\n[0,564–0,582]","0,245\n[0,243–0,246]","0,500\n± 0,000"
Logística aditiva (modo + hora),"0,748\n[0,740–0,757]","0,714\n[0,706–0,722]","0,892\n[0,883–0,902]","0,570\n[0,558–0,581]","0,695\n[0,686–0,704]","0,739\n[0,732–0,746]","0,836\n[0,829–0,844]","0,187\n[0,184–0,190]","0,746\n± 0,004"
Logística modo × jornada,"0,775\n[0,767–0,782]","0,725\n[0,717–0,733]","0,895\n[0,886–0,905]","0,588\n[0,576–0,600]","0,710\n[0,700–0,719]","0,748\n[0,741–0,755]","0,824\n[0,817–0,831]","0,175\n[0,173–0,178]","0,774\n± 0,004"


*Los IC 95 % se obtienen mediante bootstrap agrupado por pedido. Debajo de cada valor, el IC 95 % (prueba) o la desviación entre pliegues (CV temporal).*

,modelo,C elegido por CV temporal (1/regularización)
0,Dummy — clase mayoritaria / prior,NaN
1,Logística aditiva (modo + hora),10.00
2,Logística modo × jornada,0.01


**Dummy — clase mayoritaria / prior:** predice siempre la clase mayoritaria (tardía) y usa la prevalencia como probabilidad. Por eso tiene Recall = 1 y un F1 relativamente alto (0,729) sin haber aprendido ningún patrón: su ROC-AUC y su Balanced Accuracy valen 0,5.

In [3]:
e = pr.set_index("modelo")
cv_ = F.tabla("modelos_cv.csv")
cvA = cv_[(cv_["modelo"] == A) & (cv_["métrica"] == "ROC-AUC")]["media"].iloc[0]
md_out(f"""**Interpretación.**

- El **Dummy** obtiene ROC-AUC {F.num(e.loc[F.MODELO_DUMMY, 'ROC-AUC'], 1)} y Balanced Accuracy {F.num(e.loc[F.MODELO_DUMMY, 'Balanced Accuracy'], 1)} (no ordena ni distingue clases); su Accuracy ({F.num(e.loc[F.MODELO_DUMMY, 'Accuracy'])}) y su Precision ({F.num(e.loc[F.MODELO_DUMMY, 'Precision'])})
  son simplemente la proporción de tardías en la prueba. Su Recall = 1 y su F1 alto ({F.num(e.loc[F.MODELO_DUMMY, 'F1'])}) vienen de **marcar todo como tardío**, no de haber aprendido patrones; por eso F1, Accuracy y Recall
  no son la métrica principal.
- La **logística aditiva** ya ordena razonablemente ({F.num(e.loc[B_, 'ROC-AUC'])}), porque el modo de envío lleva casi toda la señal.
- La **logística modo × jornada** sube a **{F.num(e.loc[A, 'ROC-AUC'])}** y mejora ROC-AUC, Accuracy, Precision, Recall, F1, Balanced Accuracy y Brier frente a la aditiva, porque captura la hora de corte del mediodía en Same Day.
  Su Average Precision es algo menor ({F.num(e.loc[A, 'Average Precision'])} vs {F.num(e.loc[B_, 'Average Precision'])}): el modelo aditivo, al variar la probabilidad con la hora dentro de cada modo, rompe empates y eso favorece levemente a la AP, que es sensible al orden dentro de grupos con
  la misma probabilidad. La métrica principal (ROC-AUC) y la calibración favorecen a la interacción.
- **Precision frente a Recall.** Con el umbral 0,50 la logística modo × jornada tiene Precision {F.num(e.loc[A, 'Precision'])} (casi 9 de cada 10 líneas marcadas como tardías lo son) pero Recall {F.num(e.loc[A, 'Recall'])}: deja sin marcar a las líneas Standard Class, que se retrasan con una frecuencia cercana a 40 % y que con la información disponible no se pueden separar de las puntuales.
- **Comparación con la referencia empírica.** {T.LECTURAS['tabla']} Con las variables y relaciones evaluadas no se encontró señal adicional, pero eso no demuestra que ningún otro modelo pueda superarla.
- La columna de **ROC-AUC CV temporal** ({F.num(cvA, 3)}) es muy cercana a la de la prueba ({F.num(e.loc[A, 'ROC-AUC'], 3)}): el modelo generaliza al futuro, incluido el régimen de pedidos de una línea de finales de 2017.""")

**Interpretación.**

- El **Dummy** obtiene ROC-AUC 0,5 y Balanced Accuracy 0,5 (no ordena ni distingue clases); su Accuracy (0,573) y su Precision (0,573)
  son simplemente la proporción de tardías en la prueba. Su Recall = 1 y su F1 alto (0,729) vienen de **marcar todo como tardío**, no de haber aprendido patrones; por eso F1, Accuracy y Recall
  no son la métrica principal.
- La **logística aditiva** ya ordena razonablemente (0,748), porque el modo de envío lleva casi toda la señal.
- La **logística modo × jornada** sube a **0,775** y mejora ROC-AUC, Accuracy, Precision, Recall, F1, Balanced Accuracy y Brier frente a la aditiva, porque captura la hora de corte del mediodía en Same Day.
  Su Average Precision es algo menor (0,824 vs 0,836): el modelo aditivo, al variar la probabilidad con la hora dentro de cada modo, rompe empates y eso favorece levemente a la AP, que es sensible al orden dentro de grupos con
  la misma probabilidad. La métrica principal (ROC-AUC) y la calibración favorecen a la interacción.
- **Precision frente a Recall.** Con el umbral 0,50 la logística modo × jornada tiene Precision 0,895 (casi 9 de cada 10 líneas marcadas como tardías lo son) pero Recall 0,588: deja sin marcar a las líneas Standard Class, que se retrasan con una frecuencia cercana a 40 % y que con la información disponible no se pueden separar de las puntuales.
- **Comparación con la referencia empírica.** Con las variables y relaciones evaluadas, la referencia empírica basada en modo × jornada alcanza ROC-AUC ≈ 0,775 en la prueba y la regresión logística modo × jornada obtiene un resultado prácticamente equivalente (0,775). Con umbral 0,50 su Precision (0,895) es alta y su Recall (0,588) moderado. Con las variables y relaciones evaluadas no se encontró señal adicional, pero eso no demuestra que ningún otro modelo pueda superarla.
- La columna de **ROC-AUC CV temporal** (0,774) es muy cercana a la de la prueba (0,775): el modelo generaliza al futuro, incluido el régimen de pedidos de una línea de finales de 2017.

### Detalle no mostrado en el dashboard: diferencia pareada entre los dos modelos logísticos

In [4]:
dif = paired_cluster_bootstrap(pred["y"], (pred[f"p::{A}"] >= 0.5).astype(int), pred[f"p::{A}"],
                               (pred[f"p::{B_}"] >= 0.5).astype(int), pred[f"p::{B_}"], pred["order_id"], B=500)
dif.round(4)

,Diferencia,IC inferior,IC superior
Métrica,,,
Accuracy,0.0108,0.0090,0.0127
Precision,0.0031,0.0025,0.0037
Recall,0.0189,0.0158,0.0222
F1,0.0149,0.0124,0.0175
Balanced Accuracy,0.0094,0.0079,0.0111
ROC-AUC,0.0261,0.0228,0.0297
Average Precision,-0.0123,-0.0143,-0.0103
Brier,-0.0117,-0.0128,-0.0108


In [5]:
menor_es_mejor = ["Brier"]          # en Brier una diferencia negativa (A − B < 0) es una MEJORA de A
mejora = [m for m in dif.index if (dif.loc[m, "IC superior"] < 0 if m in menor_es_mejor else dif.loc[m, "IC inferior"] > 0)]
peor = [m for m in dif.index if (dif.loc[m, "IC inferior"] > 0 if m in menor_es_mejor else dif.loc[m, "IC superior"] < 0)]
md_out(f"Con los mismos pedidos remuestreados en ambos modelos, la interacción **mejora** (IC que no incluye 0, y en el sentido favorable: más alto en todas las métricas salvo Brier, donde menor es mejor) en: {', '.join(mejora)}. "
       f"**Empeora** en: {', '.join(peor) if peor else 'ninguna métrica'} (Average Precision, diferencia {F.num(dif.loc['Average Precision', 'Diferencia'], 4)}), como se explicó arriba.")

Con los mismos pedidos remuestreados en ambos modelos, la interacción **mejora** (IC que no incluye 0, y en el sentido favorable: más alto en todas las métricas salvo Brier, donde menor es mejor) en: Accuracy, Precision, Recall, F1, Balanced Accuracy, ROC-AUC, Brier. **Empeora** en: Average Precision (Average Precision, diferencia -0,0123), como se explicó arriba.

## 15.2 Discriminación: curva ROC y curva Precision–Recall (TEST)

In [6]:
mostrar(F.fig_roc())
mostrar(F.fig_pr())

**Interpretación (curva ROC, lo que muestra el dashboard).** El modelo modo × jornada (azul) está por encima del aditivo (naranja) y muy por encima de la diagonal del Dummy. La curva tiene **pocos vértices** porque el puntaje solo
toma 8 valores: cada vértice corresponde a «incluir la siguiente celda» (primero First Class y Same
Day PM, casi siempre tardíos; luego Second Class; luego Standard; al final Same Day AM). El primer
tramo vertical muestra que se puede detectar una fracción de las tardías **sin ninguna falsa alarma**
(Same Day PM y First Class), y el codo corresponde al umbral 0,5.

**Detalles que no aparecen en el dashboard — curva Precision–Recall.** Muestra, para cada nivel de Recall, qué fracción de lo marcado como tardío realmente lo es. La línea punteada es la Precision de un clasificador sin información
(la **prevalencia**): cualquier curva por debajo de ella sería peor que marcar al azar. La **Average Precision** (AP) es el área bajo esta curva calculada como suma escalonada. Como las clases están relativamente equilibradas,
la AP del Dummy ya es alta (= prevalencia), de modo que lo informativo es la **diferencia** respecto a ella. Para el modelo modo × jornada la Precision se mantiene en 1 mientras solo se marcan
las celdas casi seguras, cae al incluir Second Class y Standard y converge a la prevalencia cuando se exige Recall 1 (marcar todo). La curva escalonada del modelo elegido (8 puntos) frente a la más suave del aditivo explica
por qué la AP del aditivo es ligeramente mayor aunque su ROC-AUC sea menor: la AP es sensible al orden dentro de los empates.

In [7]:
ap = F.tabla("modelos_prueba.csv").set_index("modelo")["Average Precision"]
md_out(f"Average Precision (área bajo la curva Precision–Recall, TEST): modo × jornada {F.num(ap[A])}, aditiva {F.num(ap[B_])}, Dummy {F.num(ap[F.MODELO_DUMMY])} (= prevalencia en la prueba). "
       "El dashboard no muestra esta curva ni la AP.")

Average Precision (área bajo la curva Precision–Recall, TEST): modo × jornada 0,824, aditiva 0,836, Dummy 0,573 (= prevalencia en la prueba). El dashboard no muestra esta curva ni la AP.

## 15.3 Umbral: exploración con OOF y evaluación final en TEST

**Cambio metodológico.** La versión anterior calculaba las métricas del deslizador con las predicciones y etiquetas de la **prueba**, lo que invita a «afinar» el umbral mirando la prueba. Ahora:

1. `preparar_datos.py` genera **predicciones out-of-fold (OOF)** del entrenamiento con **los mismos pliegues temporales** de la validación cruzada (ventana creciente con gap de 7 días) y las guarda en `dashboard/datos/predicciones_oof_train.csv.gz`.
   Cada línea del entrenamiento que cae en un bloque de validación recibe la predicción de un modelo que **no la vio** al ajustarse; el primer 30 % del tiempo (el entrenamiento mínimo de la primera ventana) nunca se valida y por tanto no tiene OOF.
2. El **deslizador** del dashboard usa solo esas predicciones OOF. La nota visible dice: «*Exploración del umbral con predicciones out-of-fold del entrenamiento. El conjunto de prueba no se utiliza para seleccionar el umbral.*»
3. La prueba se evalúa **una vez, en una sección aparte y sin deslizador**, con el umbral **0,50 fijado de antemano** (el mismo del capítulo 11).

In [8]:
c = F.contexto()
md_out(f"Líneas con predicción OOF: {es(c['oof_lineas'], 0)} de {es(c['modelado_train'], 0)} del entrenamiento de modelado "
       f"(fechas {oof['order_date'].min():%Y-%m-%d} a {oof['order_date'].max():%Y-%m-%d}, anteriores al gap que empieza el {c['inicio_gap']}). Pedidos en común con la prueba: "
       f"{len(set(oof['order_id']) & set(pred['order_id']))}.")
filas = []
for u in [0.1, 0.3, 0.5, 0.7, 0.85, 0.95]:
    m = F.metricas_umbral(A, u, "oof")
    filas.append({"umbral": u, "% marcado tardío": 100 * m["marcados_tardía"], "Accuracy": m["Accuracy"], "Precision": m["Precision"],
                  "Recall": m["Recall"], "F1": m["F1"], "Balanced Accuracy": m["Balanced Accuracy"]})
md_out("**Exploración del umbral (OOF del entrenamiento; es lo que muestra el deslizador):**")
display(pd.DataFrame(filas).round(3))
mostrar(F.fig_confusion(A, 0.5, "oof"))

Líneas con predicción OOF: 95.930 de 137.043 del entrenamiento de modelado (fechas 2015-09-09 a 2017-04-14, anteriores al gap que empieza el 2017-04-15). Pedidos en común con la prueba: 0.

**Exploración del umbral (OOF del entrenamiento; es lo que muestra el deslizador):**

,umbral,% marcado tardío,Accuracy,Precision,Recall,F1,Balanced Accuracy
0,0.10,100.000,0.573,0.573,1.000,0.729,0.500
1,0.30,97.111,0.602,0.590,1.000,0.742,0.534
2,0.50,37.463,0.722,0.894,0.584,0.706,0.745
3,0.70,37.463,0.722,0.894,0.584,0.706,0.745
4,0.85,16.429,0.591,1.000,0.287,0.445,0.643
5,0.95,9.475,0.521,1.000,0.165,0.284,0.583


In [9]:
m_oof, m_test = F.metricas_umbral(A, 0.5, "oof"), F.metricas_umbral(A, F.umbral_final(), "test")
m03, m85 = F.metricas_umbral(A, 0.3, "oof"), F.metricas_umbral(A, 0.85, "oof")
md_out(f"**Interpretación (OOF).** Con umbral 0,5 el modelo marca como tardías las celdas con probabilidad ≥ 0,5 (First Class, Same Day PM y Second Class; ≈ {F.pct(m_oof['marcados_tardía'], 0)} de las líneas); "
       f"acierta el {F.pct(m_oof['cm'][0, 0] / m_oof['cm'][0].sum(), 0)} de las no tardías, pero solo detecta el {F.pct(m_oof['Recall'], 0)} de las tardías: el resto son líneas Standard que se retrasan con una frecuencia cercana a 40 % "
       f"y que ningún umbral puede separar sin marcar también a las Standard puntuales. En la tabla, al bajar el umbral a 0,3 se marca todo Standard ({F.pct(m03['marcados_tardía'], 0)} de las líneas) y el Recall sube a {F.num(m03['Recall'], 2)} a costa de la Precision ({F.num(m03['Precision'], 2)}) y de muchas falsas alarmas; "
       f"al subirlo a 0,85 el modelo solo marca las celdas casi seguras (Precision {F.num(m85['Precision'], 2)}, Recall {F.num(m85['Recall'], 2)}). **El ROC-AUC no cambia** con el umbral: el umbral es una decisión operativa (capacidad para intervenir y costo de cada error), no una propiedad del modelo.")

**Interpretación (OOF).** Con umbral 0,5 el modelo marca como tardías las celdas con probabilidad ≥ 0,5 (First Class, Same Day PM y Second Class; ≈ 37 % de las líneas); acierta el 91 % de las no tardías, pero solo detecta el 58 % de las tardías: el resto son líneas Standard que se retrasan con una frecuencia cercana a 40 % y que ningún umbral puede separar sin marcar también a las Standard puntuales. En la tabla, al bajar el umbral a 0,3 se marca todo Standard (97 % de las líneas) y el Recall sube a 1,00 a costa de la Precision (0,59) y de muchas falsas alarmas; al subirlo a 0,85 el modelo solo marca las celdas casi seguras (Precision 1,00, Recall 0,29). **El ROC-AUC no cambia** con el umbral: el umbral es una decisión operativa (capacidad para intervenir y costo de cada error), no una propiedad del modelo.

### Evaluación final en TEST con umbral previamente fijado = 0,50

In [10]:
mostrar(F.fig_confusion(A, F.umbral_final(), "test"))
final = pd.DataFrame([{"origen": "OOF del entrenamiento", **{k: m_oof[k] for k in ["Accuracy", "Precision", "Recall", "F1", "Balanced Accuracy"]}},
                      {"origen": "TEST (umbral 0,50 fijo)", **{k: m_test[k] for k in ["Accuracy", "Precision", "Recall", "F1", "Balanced Accuracy"]}}])
display(final.round(3))
umb11 = pd.read_csv(TABLES / "umbral_alternativo.csv", index_col=0)
md_out("Las métricas con el umbral 0,50 son parecidas en OOF y en TEST: el umbral elegido de antemano no depende de la prueba. En el capítulo 11 el umbral alternativo de Youden también se obtuvo con OOF "
       f"({umb11.index[1]}) y mostró que cambiarlo mueve Accuracy, Recall y F1 sin alterar el ROC-AUC.")

,origen,Accuracy,Precision,Recall,F1,Balanced Accuracy
0,OOF del entrenamiento,0.722,0.894,0.584,0.706,0.745
1,"TEST (umbral 0,50 fijo)",0.725,0.895,0.588,0.710,0.748


Las métricas con el umbral 0,50 son parecidas en OOF y en TEST: el umbral elegido de antemano no depende de la prueba. En el capítulo 11 el umbral alternativo de Youden también se obtuvo con OOF (umbral Youden OOF = 0.806) y mostró que cambiarlo mueve Accuracy, Recall y F1 sin alterar el ROC-AUC.

In [11]:
md_out(f"""**Interpretación (TEST, umbral 0,50 fijado de antemano).** El modelo marca como tardía el {F.pct(m_test['marcados_tardía'], 0)} de las líneas de la prueba; acierta el {F.pct(m_test['cm'][0, 0] / m_test['cm'][0].sum(), 0)} de las no tardías
({es(m_test['cm'][0, 1], 0)} falsas alarmas) y detecta el {F.pct(m_test['Recall'], 0)} de las tardías ({es(m_test['cm'][1, 0], 0)} no detectadas). Es la lectura honesta del desempeño operativo: se decidió el umbral sin mirar la prueba y aquí solo se evalúa.""")

**Interpretación (TEST, umbral 0,50 fijado de antemano).** El modelo marca como tardía el 38 % de las líneas de la prueba; acierta el 91 % de las no tardías
(1.362 falsas alarmas) y detecta el 59 % de las tardías (8.155 no detectadas). Es la lectura honesta del desempeño operativo: se decidió el umbral sin mirar la prueba y aquí solo se evalúa.

**Detalles que no aparecen en el dashboard.** El dashboard muestra la matriz de confusión, el porcentaje marcado como tardío y el ROC-AUC (que no cambia con el umbral). Aquí además están la tabla de exploración
(Accuracy, Precision, Recall, F1 y Balanced Accuracy para cada umbral), la comparación OOF frente a TEST con el mismo umbral y el vínculo con el umbral alternativo de Youden del capítulo 11.

## 15.4 Calibración corregida (TEST)

**Qué estaba mal.** La versión anterior agrupaba con `pd.qcut(prediccion.rank(method="first"), 10)`. Con `rank(method="first")` los empates se rompen **por orden de aparición**, así que observaciones con **exactamente la misma probabilidad**
se repartían entre deciles distintos de forma arbitraria y, a la inversa, un mismo decil podía **mezclar probabilidades distintas**. El modelo modo × jornada solo produce 8 probabilidades distintas: por eso, con deciles, varios grupos repetían casi la misma probabilidad media y otros
promediaban dos probabilidades diferentes en un valor que el modelo nunca produce (ver la tabla del método anterior más abajo).

**Cómo se corrige.** `figuras._bins_calibracion()` **no parte empates**: si el modelo genera pocas probabilidades distintas (≤ 20), cada probabilidad única es un grupo; si genera muchas (modelo aditivo), se usan deciles sobre los **valores** (`qcut` con `duplicates="drop"`),
de modo que dos observaciones con el mismo valor siempre caen en el mismo grupo. Se grafica **probabilidad media predicha vs frecuencia observada** y el hover muestra el tamaño *n* de cada grupo.

In [12]:
mostrar(F.fig_calibracion())
for m in [A, B_]:
    g = F._bins_calibracion(pred[f"p::{m}"], pred["y"]).rename(columns={"pm": "prob. media predicha", "obs": "frecuencia observada", "n": "n líneas"})
    md_out(f"**{m}** — {pred[f'p::{m}'].round(6).nunique()} probabilidades distintas → {len(g)} grupos:")
    display(g.round(3).reset_index(drop=True))
# Comparación con el método anterior (deciles por rango): empates partidos arbitrariamente
ant = (pred.assign(decil=pd.qcut(pred[f"p::{A}"].rank(method="first"), 10, labels=False))
           .groupby("decil").agg(prob_media=(f"p::{A}", "mean"), frecuencia_observada=("y", "mean"), n=("y", "size")))
md_out("**Método anterior (deciles por rango) en el modelo modo × jornada**: varios deciles repiten casi la misma probabilidad (empates partidos) y otro promedia dos probabilidades distintas en un valor que el modelo no genera; son artefactos del método, no del modelo.")
display(ant.round(3))
brier = F.tabla("modelos_prueba.csv").set_index("modelo")["Brier"]
prev = pred["y"].mean()
md_out(f"Brier: modo × jornada {F.num(brier[A])}, aditiva {F.num(brier[B_])}, Dummy {F.num(brier[F.MODELO_DUMMY])} (= prevalencia × (1 − prevalencia) ≈ {F.num(prev * (1 - prev))}).")

**Logística modo × jornada** — 8 probabilidades distintas → 8 grupos:

,prob. media predicha,frecuencia observada,n líneas
0,0.112,0.000,878
1,0.403,0.394,10525
2,0.407,0.394,10156
3,0.804,0.808,3286
4,0.807,0.789,3457
5,0.912,1.000,2784
6,0.970,1.000,987
7,0.983,1.000,2510


**Logística aditiva (modo + hora)** — 93 probabilidades distintas → 10 grupos:

,prob. media predicha,frecuencia observada,n líneas
0,0.357,0.385,4311
1,0.369,0.394,2681
2,0.387,0.406,3533
3,0.410,0.412,3400
4,0.430,0.343,3661
5,0.442,0.377,3574
6,0.651,0.773,3229
7,0.803,0.790,3479
8,0.929,0.914,3490
9,1.000,1.000,3225


**Método anterior (deciles por rango) en el modelo modo × jornada**: varios deciles repiten casi la misma probabilidad (empates partidos) y otro promedia dos probabilidades distintas en un valor que el modelo no genera; son artefactos del método, no del modelo.

,prob_media,frecuencia_observada,n
decil,,,
0,0.329,0.295,3459
1,0.403,0.390,3458
2,0.403,0.398,3458
3,0.406,0.391,3458
4,0.407,0.396,3459
5,0.407,0.392,3458
6,0.711,0.714,3458
7,0.806,0.792,3458
8,0.893,0.962,3458


Brier: modo × jornada 0,175, aditiva 0,187, Dummy 0,245 (= prevalencia × (1 − prevalencia) ≈ 0,245).

In [13]:
md_out("**Interpretación.** " + T.LECTURAS["calibracion"] + " Los puntos del modelo elegido siguen la diagonal: cuando predice ≈ 40 % se retrasa ≈ 40 % (Standard) y cuando predice ≈ 80 %, ≈ 80 % (Second Class). "
       "En los extremos la regularización (C = 0,01) encoge un poco las probabilidades (predice ≈ 0,9 donde la frecuencia observada es 1 y ≈ 0,1 donde es 0), sin cambiar el orden. "
       "El modelo aditivo se desvía más porque mezcla Same Day AM y PM en una sola probabilidad. Unas probabilidades razonablemente calibradas permiten usar el puntaje como estimación del riesgo de retraso para planificar; "
       "la calibración se evalúa en la prueba pero no se usa para seleccionar nada.")

**Interpretación.** Para el modelo modo × jornada cada probabilidad predicha distinta es un punto (los empates no se parten en deciles): la mayor diferencia entre probabilidad media predicha y frecuencia observada es de 11,2 puntos porcentuales. Los puntos del modelo elegido siguen la diagonal: cuando predice ≈ 40 % se retrasa ≈ 40 % (Standard) y cuando predice ≈ 80 %, ≈ 80 % (Second Class). En los extremos la regularización (C = 0,01) encoge un poco las probabilidades (predice ≈ 0,9 donde la frecuencia observada es 1 y ≈ 0,1 donde es 0), sin cambiar el orden. El modelo aditivo se desvía más porque mezcla Same Day AM y PM en una sola probabilidad. Unas probabilidades razonablemente calibradas permiten usar el puntaje como estimación del riesgo de retraso para planificar; la calibración se evalúa en la prueba pero no se usa para seleccionar nada.

**Detalles que no aparecen en el dashboard.** El dashboard grafica probabilidad media predicha frente a frecuencia observada. Aquí además: el tamaño *n* de cada grupo (tablas de arriba), la comparación con el método anterior
de deciles por rango (que partía empates) y el Brier de cada modelo, que resume calibración y discriminación en un solo número.

## 15.5 Probabilidad por modo × jornada (TEST)

In [14]:
mostrar(F.fig_celdas())
coef = F.tabla("modelos_coeficientes.csv")
coef[coef["modelo"] == F.MODELO_ELEGIDO].assign(variable=lambda d: d["variable"].str.replace("cat__mode_x_daypart_", "")).round(4)

,modelo,variable,coeficiente,odds_ratio
5,Logística modo × jornada,First Class | PM,1.7545,5.7807
6,Logística modo × jornada,Same Day | AM,-4.4040,0.0122
7,Logística modo × jornada,Same Day | PM,1.1331,3.1052
8,Logística modo × jornada,Second Class | AM,-0.9213,0.3980
9,Logística modo × jornada,Second Class | PM,-0.9040,0.4049
10,Logística modo × jornada,Standard Class | AM,-2.7253,0.0655
11,Logística modo × jornada,Standard Class | PM,-2.7075,0.0667


**Interpretación.** Es la forma más clara de leer el modelo: cada barra gris es la tasa observada en entrenamiento y el rombo azul es la probabilidad promedio que asigna el modelo en la prueba; coinciden.
**El modelo reproduce las diferencias observadas en entrenamiento entre las combinaciones de modo de envío y jornada:** First Class y Same Day de la tarde con tasas muy altas, Second Class ≈ 80 %, Standard ≈ 40 % y Same Day de la mañana con la tasa más baja.
Esto describe asociaciones observadas, no relaciones causales.

**No aparece en el dashboard:** los coeficientes (tabla) se expresan frente a la categoría de
referencia First Class | AM y su magnitud depende de la regularización por la separación casi
completa (capítulo 11); por eso el dashboard muestra probabilidades y no odds ratios.
**Implicación operativa (hipótesis a validar):** las dos acciones con más potencial no requieren un modelo complejo:
revisar la promesa de First Class y comunicar la hora de corte de Same Day.

## 15.6 Diagnóstico (I): desempeño por subgrupos (TEST)

In [15]:
for v in ["market", "customer_segment", "periodo", "shipping_mode"]:
    mostrar(F.fig_subgrupos(v))
filas = []
for v in ["market", "customer_segment", "periodo"]:
    for g, d in pred.groupby(v):
        filas.append({"variable": v, "grupo": g, "n": len(d), "ROC-AUC (modo × jornada)": skm.roc_auc_score(d["y"], d[f"p::{A}"])})
sub = pd.DataFrame(filas)
display(sub.round(3))
rango = (sub["ROC-AUC (modo × jornada)"].min(), sub["ROC-AUC (modo × jornada)"].max())

,variable,grupo,n,ROC-AUC (modo × jornada)
0,market,Europe,19943,0.779
1,market,LATAM,9260,0.765
2,market,Pacific Asia,5380,0.775
3,customer_segment,Consumer,17876,0.771
4,customer_segment,Corporate,10497,0.778
5,customer_segment,Home Office,6210,0.780
6,periodo,abr–sep 2017,26390,0.774
7,periodo,desde oct-2017,8193,0.776


In [16]:
md_out(f"**Interpretación.** {T.LECTURAS['subgrupos']} Entre mercados, segmentos de cliente y periodos (antes y después del cambio de régimen de octubre de 2017) el ROC-AUC del modelo elegido varía entre {F.num(rango[0], 2)} y {F.num(rango[1], 2)}; "
       "eso describe **estabilidad del desempeño**, no equidad: no se evaluaron tasas de error por grupo (falsos positivos y negativos), calibración por grupo ni el efecto de decisiones sobre cada grupo. "
       "El desempeño tampoco parece degradarse en el régimen nuevo, coherente con que modo y jornada son estables (PSI ≈ 0). **Dentro de un mismo modo** el ROC-AUC baja a ≈ 0,5 en Standard y Second Class "
       "(el modelo no tiene información para ordenar dentro de ellos) y sube a 1 en Same Day (la jornada lo determina); First Class no aparece porque solo tiene una clase.")

**Interpretación.** El ROC-AUC observado es similar entre los subgrupos evaluados. Este análisis describe estabilidad del desempeño y no constituye una auditoría completa de equidad o fairness. Entre mercados, segmentos de cliente y periodos (antes y después del cambio de régimen de octubre de 2017) el ROC-AUC del modelo elegido varía entre 0,76 y 0,78; eso describe **estabilidad del desempeño**, no equidad: no se evaluaron tasas de error por grupo (falsos positivos y negativos), calibración por grupo ni el efecto de decisiones sobre cada grupo. El desempeño tampoco parece degradarse en el régimen nuevo, coherente con que modo y jornada son estables (PSI ≈ 0). **Dentro de un mismo modo** el ROC-AUC baja a ≈ 0,5 en Standard y Second Class (el modelo no tiene información para ordenar dentro de ellos) y sube a 1 en Same Day (la jornada lo determina); First Class no aparece porque solo tiene una clase.

In [17]:
md_out(f"""**Detalles que no aparecen en el dashboard.** La tabla anterior da el tamaño y el ROC-AUC exacto de cada uno de los {len(sub)} grupos (mercado, segmento, periodo); el grupo más pequeño tiene {es(sub['n'].min(), 0)} líneas
y, con pocas líneas, el ROC-AUC es menos estable. No se calcularon intervalos de confianza por grupo.""")

**Detalles que no aparecen en el dashboard.** La tabla anterior da el tamaño y el ROC-AUC exacto de cada uno de los 8 grupos (mercado, segmento, periodo); el grupo más pequeño tiene 5.380 líneas
y, con pocas líneas, el ROC-AUC es menos estable. No se calcularon intervalos de confianza por grupo.

## 15.7 Diagnóstico (II): selección de variables y curva de aprendizaje

In [18]:
mostrar(F.fig_seleccion())
mostrar(F.fig_curva_aprendizaje())
display(F.tabla("diagnostico_residuos.csv").round(4))

,diagnóstico,estadístico,p
0,"Ljung-Box residuo diario, rezago 7",5.1598,0.6405
1,"Ljung-Box residuo diario, rezago 14",13.5084,0.4869
2,"Ljung-Box residuo diario, rezago 28",36.9455,0.1201
3,I de Moran del residuo por celda (132 celdas),-0.0099,0.9120


In [19]:
md_out(f"""**Interpretación.**

- **Selección:** pasar del modo solo (A) a modo + hora aditivo (B) mejora poco; la interacción (C) da el salto. {T.LECTURAS['seleccion']} Esas variables adicionales (producto, pago, calendario, carga, destino, coordenadas: conjuntos D–F)
  tienen AUC univariado cercano a 0,5. Por la regla de una desviación estándar se eligió el conjunto más simple (C).
- **Curva de aprendizaje:** {T.LECTURAS['aprendizaje']} (El tamaño se define como el primero cuyo ROC-AUC de validación queda a menos de 0,005 del máximo, redondeado a miles de 5 en 5; la curva termina en el 100 % del entrenamiento, así que no se observa qué pasaría con más datos.)
- **Residuos (no mostrado en el dashboard):** el residuo medio diario no tiene autocorrelación (Ljung-Box) y el residuo por celda espacial no tiene I de Moran significativa en las pruebas realizadas: no se detecta estructura temporal ni espacial sin capturar.""")

**Interpretación.**

- **Selección:** pasar del modo solo (A) a modo + hora aditivo (B) mejora poco; la interacción (C) da el salto. Añadir producto, cliente, destino o coordenadas no mejora el ROC-AUC de validación cruzada respecto de modo × jornada (diferencias dentro de la variación entre pliegues); se elige el conjunto más simple. Esas variables adicionales (producto, pago, calendario, carga, destino, coordenadas: conjuntos D–F)
  tienen AUC univariado cercano a 0,5. Por la regla de una desviación estándar se eligió el conjunto más simple (C).
- **Curva de aprendizaje:** Las curvas de entrenamiento y validación son cercanas y la mejora se estabiliza aproximadamente desde 40.000 líneas; para esta especificación no se observa evidencia importante de sobreajuste y el beneficio marginal de aumentar la muestra parece pequeño. (El tamaño se define como el primero cuyo ROC-AUC de validación queda a menos de 0,005 del máximo, redondeado a miles de 5 en 5; la curva termina en el 100 % del entrenamiento, así que no se observa qué pasaría con más datos.)
- **Residuos (no mostrado en el dashboard):** el residuo medio diario no tiene autocorrelación (Ljung-Box) y el residuo por celda espacial no tiene I de Moran significativa en las pruebas realizadas: no se detecta estructura temporal ni espacial sin capturar.

## 15.8 Conclusión del Entregable 2

El dashboard comunica, en tres pestañas, **qué problema se resuelve y por qué importa**, **qué dicen los datos** y **qué logra el modelo base**. El mensaje central es consistente en las tres:
sin fuga, el riesgo de retraso en DataCo depende de **la promesa de entrega (modo de envío)** y,
en Same Day, de la **hora de corte del mediodía**. Una regresión logística con esa interacción
alcanza un ROC-AUC prácticamente igual al de la referencia empírica sin fuga (tabla de tasas modo × jornada), está razonablemente calibrada, generaliza al futuro en la validación y la prueba cronológicas y su desempeño
es similar entre los mercados, segmentos y periodos evaluados. Con las variables y relaciones evaluadas, el resto del retraso (Standard y Second Class) no mostró señal adicional: eso sugiere que el límite
está en la información disponible, aunque no se demuestra que ningún modelo posterior pueda superar esta referencia.

Dos cautelas finales: (1) el análisis es a nivel de **línea de pedido**; (2) los resultados dependen de que el dataset es simulado y de la partición cronológica elegida.